In [2]:
"""
Bristol Road Network — Safe Routing Service (Heat Method)
=========================================================
Geodesic routing that avoids crime hotspots, stays near CCTV cameras,
prefers lit streets, and minimises fast/dangerous roads.
 
Safety priority order (highest → lowest):
  1. Avoid high-crime areas      (w_crime)
  2. Stay near CCTV cameras      (w_cctv)
  3. Prefer lit streets          (w_lighting)
  4. Avoid fast roads / danger   (w_safety_risk)
 
Data sources expected
---------------------
  crime_csv  : data.police.uk CSV — must have columns "Latitude", "Longitude"
               (download Avon & Somerset, select date range, unzip)
 
  cctv_csv   : Bristol Open Data CCTV CSV — must have columns
               "latitude" / "longitude"  OR  "Latitude" / "Longitude"
               (https://opendata.bristol.gov.uk → search "CCTV")
 
  lights_csv : Bristol Open Data Street Lighting CSV — must have columns
               "latitude" / "longitude"
               (https://opendata.bristol.gov.uk → search "Street lighting")
               If you omit this, the service falls back to OSM lit= tags only.
 
Quick start
-----------
    service = RoutingService(
        crime_csv  = "avon_somerset_crime.csv",
        cctv_csv   = "bristol_cctv.csv",
        lights_csv = "bristol_street_lighting.csv",   # optional
    )
 
    result = service.query(
        origin_lat=51.4545, origin_lon=-2.5879,
        dest_lat  =51.4655, dest_lon  =-2.6020,
        profile   =PROFILES["safest_night"],
    )
    print(result)
 
    # Interactive HTML map (open in any browser)
    service.visualise_interactive(
        origin_lat=51.4545, origin_lon=-2.5879,
        dest_lat  =51.4655, dest_lon  =-2.6020,
        profile   =PROFILES["safest_night"],
    )
 
Dependencies
------------
    pip install osmnx networkx numpy scipy matplotlib folium branca pandas
"""
 
from __future__ import annotations
 
import hashlib
import json
import os
import time
from dataclasses import dataclass, field
from typing import Optional
 
import folium
import branca.colormap as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from scipy.sparse import csr_matrix, diags, lil_matrix
from scipy.sparse.linalg import spsolve
from scipy.stats import gaussian_kde
from scipy.spatial import cKDTree
 
 
# ══════════════════════════════════════════════════════════════════════════════
# USER PROFILE  — weights & hard constraints
# ══════════════════════════════════════════════════════════════════════════════
 
@dataclass
class UserProfile:
    """
    Per-request routing preferences.
 
    Soft weights (0.0–1.0)
    ----------------------
    All additive cost penalties; 0 = ignore, 1 = maximum avoidance.
 
    Priority order for safe routing:
      1. w_crime        — avoid high-crime density areas       (highest)
      2. w_cctv         — stay within reach of CCTV cameras
      3. w_lighting     — prefer lit streets
      4. w_safety_risk  — avoid fast / dangerous roads         (lowest)
 
    Hard constraints (bool)
    -----------------------
    Set edge cost to infinity — solver will never route through them.
    """
 
    WALKING_SPEED_KMH: float = 5.0
    WALKING_distance_min: float = (5.0 / 60) * 1000   # metres per minute
 
    # ── Soft weights ────────────────────────────────────────────────────────
    w_distance   : float = 0.3
    w_travel_time: float = 0.3
 
    # Safety weights — tuned to your priority order
    w_crime      : float = 1.0   # 1. Avoid high-crime areas  ★★★★
    w_cctv       : float = 0.7   # 2. Stay near CCTV           ★★★
    w_lighting   : float = 0.5   # 3. Prefer lit streets       ★★
    w_safety_risk: float = 0.2   # 4. Avoid fast roads         ★
 
    # ── Hard constraints ────────────────────────────────────────────────────
    require_lit           : bool = False   # block every unlit edge
    require_smooth_surface: bool = False   # block cobble/unpaved
    wheelchair_accessible : bool = False   # block steps / stairs
 
    # ── Network type ────────────────────────────────────────────────────────
    network_type: str = "walk"
 
    def profile_key(self) -> str:
        """Stable 12-char hash used as part of the cache key."""
        d = {k: v for k, v in self.__dict__.items()}
        return hashlib.md5(
            json.dumps(d, sort_keys=True).encode()
        ).hexdigest()[:12]
 
 
# ── Ready-made profiles for common use cases ──────────────────────────────────
 
PROFILES: dict[str, UserProfile] = {
    # Night walk — maximum safety, hard-filter unlit streets
    "safest_night": UserProfile(
        w_distance=0.15, w_travel_time=0.15,
        w_crime=1.0, w_cctv=0.8, w_lighting=0.9, w_safety_risk=0.3,
        require_lit=True,
    ),
    # Daytime safe route — crime + CCTV matter, lighting less so
    "safest_day": UserProfile(
        w_distance=0.25, w_travel_time=0.25,
        w_crime=1.0, w_cctv=0.6, w_lighting=0.2, w_safety_risk=0.2,
    ),
    # Balanced — safety-aware but not at the cost of very long detours
    "balanced": UserProfile(
        w_distance=0.4, w_travel_time=0.4,
        w_crime=0.6, w_cctv=0.35, w_lighting=0.3, w_safety_risk=0.15,
    ),
    # Speed-first — minimal safety weighting
    "fastest": UserProfile(
        w_distance=0.3, w_travel_time=0.7,
        w_crime=0.1, w_cctv=0.0, w_lighting=0.0, w_safety_risk=0.0,
    ),
    # Wheelchair — smooth surface, no steps, avoids steep grades
    "wheelchair": UserProfile(
        w_distance=0.4, w_travel_time=0.4,
        w_crime=0.8, w_cctv=0.5,
        wheelchair_accessible=True,
        require_smooth_surface=True,
    ),
}
 
 
# ══════════════════════════════════════════════════════════════════════════════
# SURFACE PENALTY TABLE
# ══════════════════════════════════════════════════════════════════════════════
 
_SURFACE_PENALTY: dict[str, float] = {
    "asphalt"    : 0.0,
    "paved"      : 0.0,
    "concrete"   : 0.05,
    "sett"       : 0.4,
    "cobblestone": 0.5,
    "gravel"     : 0.6,
    "unpaved"    : 0.7,
    "dirt"       : 0.8,
    "grass"      : 0.9,
}
_IMPASSABLE_SURFACES = {"cobblestone", "sett", "gravel", "unpaved", "dirt", "grass"}
 
 
def _get_str(edge_data: dict, key: str, default: str = "") -> str:
    val = edge_data.get(key, default)
    if isinstance(val, list):
        return val[0] if val else default
    return str(val) if val is not None else default
 
 
# ══════════════════════════════════════════════════════════════════════════════
# COST FUNCTION  — called for every edge during the Heat Method solve
# ══════════════════════════════════════════════════════════════════════════════
 
def cost_function(edge_data: dict, profile: UserProfile) -> float:
    """
    Returns the traversal cost of an edge under the given UserProfile.
 
    Edge attributes used
    --------------------
    Standard OSM / osmnx:
        length, travel_time, surface, lit, highway, grade
 
    Injected by enrich_graph_with_safety_data():
        crime_score   ∈ [0, 1]  — 0 = safe area, 1 = crime hotspot
        cctv_penalty  ∈ [0, 1]  — 0 = right next to a camera, 1 = far away
        light_penalty ∈ [0, 1]  — 0 = near a street light, 1 = dark (augments OSM lit=)
    """
    surface = _get_str(edge_data, "surface", "asphalt")
    lit     = _get_str(edge_data, "lit", "no")
    highway = _get_str(edge_data, "highway", "")
 
    # ── 1. Hard constraints (impassable) ────────────────────────────────────
    if profile.require_smooth_surface and surface in _IMPASSABLE_SURFACES:
        return float("inf")
    if profile.require_lit and lit == "no" and edge_data.get("light_penalty", 1.0) > 0.8:
        return float("inf")
    if profile.wheelchair_accessible and highway in ("steps", "stairs"):
        return float("inf")
 
    # ── 2. Base costs ────────────────────────────────────────────────────────
    length      = float(edge_data.get("length", 100))
    travel_time = float(edge_data.get("travel_time", 30.0))
    gradient    = min(abs(float(edge_data.get("grade", 0.0))), 1.0)
 
    # ── 3. Safety scores (injected by enrichment; defaults = pessimistic) ───
    crime_score  = float(edge_data.get("crime_score",  0.5))  # default: medium risk
    cctv_penalty = float(edge_data.get("cctv_penalty", 1.0))  # default: no camera
 
    # Lighting: combine OSM lit= tag with Bristol Open Data light_penalty
    osm_lit_pen   = 0.0 if lit == "yes" else 1.0
    data_light_pen = float(edge_data.get("light_penalty", osm_lit_pen))
    # Take the minimum (most optimistic) of the two signals
    lighting_pen  = min(osm_lit_pen, data_light_pen)
 
    # Speed / danger proxy: normalise max speed (default 30 kph → 0.3)
    speed_kph    = float(edge_data.get("speed_kph", 30.0))
    safety_risk  = min(speed_kph / 100.0, 1.0)
 
    surface_pen  = _SURFACE_PENALTY.get(surface, 0.0)
 
    # ── 4. Weighted sum ──────────────────────────────────────────────────────
    cost = (
        profile.w_distance    * (length / profile.WALKING_distance_min)
      + profile.w_travel_time * (travel_time / 60.0)
      + profile.w_crime       * crime_score          # ★ highest priority
      + profile.w_cctv        * cctv_penalty         # ★★
      + profile.w_lighting    * lighting_pen         # ★★★
      + profile.w_safety_risk * safety_risk          # ★★★★
      + getattr(profile, "w_gradient", 0.0) * gradient
      + getattr(profile, "w_surface",  0.0) * surface_pen
    )
 
    return max(cost, 1e-6)
 
 
# ══════════════════════════════════════════════════════════════════════════════
# DATA ENRICHMENT  — stamp safety scores onto graph edges
# ══════════════════════════════════════════════════════════════════════════════
 
def _normalise_latlon_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Accept mixed-case latitude/longitude column names."""
    col_map = {}
    for col in df.columns:
        low = col.lower()
        if low in ("latitude", "lat"):
            col_map[col] = "lat"
        elif low in ("longitude", "lon", "long"):
            col_map[col] = "lon"
    return df.rename(columns=col_map)
 
 
def enrich_graph_with_safety_data(
    G,
    crime_csv  : str,
    cctv_csv   : str,
    lights_csv : Optional[str] = None,
    cctv_radius_m  : float = 150.0,   # "near a camera" threshold
    light_radius_m : float = 50.0,    # "near a street light" threshold
    crime_bw       : float = 0.008,   # KDE bandwidth (~900 m at Bristol latitude)
) -> object:
    """
    Stamp three safety attributes onto every edge in G:
 
        crime_score   ∈ [0, 1]  — normalised crime KDE density
        cctv_penalty  ∈ [0, 1]  — 0 right under a camera, 1 = far away
        light_penalty ∈ [0, 1]  — 0 under a street light, 1 = dark
                                   (augments OSM lit= tag in cost_function)
 
    Parameters
    ----------
    G             : OSMnx graph (modified in-place and returned)
    crime_csv     : Path to data.police.uk CSV
    cctv_csv      : Path to Bristol Open Data CCTV CSV
    lights_csv    : Path to Bristol Open Data street lighting CSV (optional)
    cctv_radius_m : Distance (metres) considered "near" a CCTV camera
    light_radius_m: Distance (metres) considered "near" a street light
    crime_bw      : Gaussian KDE bandwidth (degrees); ~0.008 ≈ 900 m
    """
    print("  Enriching graph with safety data ...")
 
    # ── 1. Collect edge midpoints ────────────────────────────────────────────
    edges     = list(G.edges(data=True))
    mid_lons  = np.array([(G.nodes[u]["x"] + G.nodes[v]["x"]) / 2
                           for u, v, _ in edges])
    mid_lats  = np.array([(G.nodes[u]["y"] + G.nodes[v]["y"]) / 2
                           for u, v, _ in edges])
    mid_arr   = np.column_stack([mid_lons, mid_lats])
 
    # Approximate metres-per-degree at Bristol (~51.45 °N)
    LAT_REF  = 51.45
    M_PER_DEG_LAT = 111_320
    M_PER_DEG_LON = 111_320 * np.cos(np.radians(LAT_REF))
 
    def deg_to_m(dist_deg_lon, dist_deg_lat=None):
        """Convert degree distance to approximate metres."""
        if dist_deg_lat is None:
            # isotropic: dist_deg_lon is a raw degree distance from cKDTree
            # approximate using mean of lat/lon scale
            avg_scale = (M_PER_DEG_LAT + M_PER_DEG_LON) / 2
            return dist_deg_lon * avg_scale
        dx = dist_deg_lon * M_PER_DEG_LON
        dy = dist_deg_lat * M_PER_DEG_LAT
        return np.sqrt(dx**2 + dy**2)
 
    # ── 2. Crime KDE ─────────────────────────────────────────────────────────
    print("    Loading crime data ...")
    crime_df = pd.read_csv(crime_csv).pipe(_normalise_latlon_columns)
    crime_df = crime_df[["lat", "lon"]].dropna()
 
    # Filter to rough Bristol bounding box (speeds up KDE)
    crime_df = crime_df[
        (crime_df["lat"].between(51.38, 51.56)) &
        (crime_df["lon"].between(-2.72, -2.48))
    ]
 
    if len(crime_df) < 5:
        print("    ⚠  Very few crime points found — check column names / bounding box.")
        crime_scores = np.full(len(edges), 0.5)
    else:
        kde = gaussian_kde(
            np.vstack([crime_df["lon"].values, crime_df["lat"].values]),
            bw_method=crime_bw,
        )
        kde_vals     = kde(mid_arr.T)
        kde_max      = kde_vals.max() or 1.0
        crime_scores = (kde_vals / kde_max).astype(float)
    print(f"    Crime: {len(crime_df):,} incidents loaded.")
 
    # ── 3. CCTV KD-tree ──────────────────────────────────────────────────────
    print("    Loading CCTV data ...")
    cctv_df = pd.read_csv(cctv_csv).pipe(_normalise_latlon_columns)
    cctv_df = cctv_df[["lat", "lon"]].dropna()
    cctv_tree   = cKDTree(np.column_stack([cctv_df["lon"], cctv_df["lat"]]))
    cctv_dists_deg, _ = cctv_tree.query(mid_arr)
    cctv_dists_m      = cctv_dists_deg * (M_PER_DEG_LAT + M_PER_DEG_LON) / 2
    cctv_penalties    = np.clip(cctv_dists_m / cctv_radius_m, 0.0, 1.0)
    print(f"    CCTV: {len(cctv_df):,} cameras loaded.")
 
    # ── 4. Street lighting KD-tree (optional) ────────────────────────────────
    if lights_csv and os.path.exists(lights_csv):
        print("    Loading street lighting data ...")
        lights_df = pd.read_csv(lights_csv).pipe(_normalise_latlon_columns)
        lights_df = lights_df[["lat", "lon"]].dropna()
        lights_tree      = cKDTree(np.column_stack([lights_df["lon"], lights_df["lat"]]))
        light_dists_deg, _ = lights_tree.query(mid_arr)
        light_dists_m      = light_dists_deg * (M_PER_DEG_LAT + M_PER_DEG_LON) / 2
        light_penalties    = np.clip(light_dists_m / light_radius_m, 0.0, 1.0)
        print(f"    Lights: {len(lights_df):,} street lights loaded.")
    else:
        # No lighting CSV: fall back to OSM lit= tag inside cost_function
        light_penalties = np.ones(len(edges))
        if lights_csv:
            print(f"    ⚠  lights_csv not found: {lights_csv} — using OSM lit= only.")
 
    # ── 5. Stamp onto edges ───────────────────────────────────────────────────
    for i, (u, v, data) in enumerate(edges):
        data["crime_score"]  = float(crime_scores[i])
        data["cctv_penalty"] = float(cctv_penalties[i])
        data["light_penalty"]= float(light_penalties[i])
 
    print(f"  ✓ Enriched {G.number_of_edges():,} edges.\n")
    return G
 
 
# ══════════════════════════════════════════════════════════════════════════════
# GRAPH BUILDER
# ══════════════════════════════════════════════════════════════════════════════
 
_EXTRA_TAGS = ["surface", "lit", "width", "grade", "maxspeed"]
 
 
def _build_graph(network_type: str = "walk",
                 cache_path: str = "./bristol_graph.graphml"):
    """
    Download (once) or load the Bristol OSMnx graph.
    Edge metadata (surface, lit, speed) is preserved for the cost function.
    Safety scores are NOT baked in here — they are added by enrich_graph_with_safety_data().
    """
    if os.path.exists(cache_path):
        print(f"  Loading graph from disk: {cache_path}")
        G = ox.load_graphml(cache_path)
    else:
        print(f"  Downloading Bristol graph (network={network_type}) ...")
        ox.settings.useful_tags_way = list(
            set(ox.settings.useful_tags_way + _EXTRA_TAGS)
        )
        G = ox.graph_from_place("Bristol, England", network_type=network_type)
        G = ox.add_edge_speeds(G)
        G = ox.add_edge_travel_times(G)
        ox.save_graphml(G, cache_path)
 
        try:
            G = ox.elevation.add_node_elevations_google(G, api_key="YOUR_KEY")
            G = ox.elevation.add_edge_grades(G)
        except Exception:
            pass   # elevation optional
 
        print(f"  Saved to {cache_path}")
 
    print(f"  Graph ready: {len(G.nodes):,} nodes / {len(G.edges):,} edges\n")
    return G
 
 
# ══════════════════════════════════════════════════════════════════════════════
# HEAT METHOD  — geodesic distance solver
# ══════════════════════════════════════════════════════════════════════════════
 
def _build_laplacian(G, profile: UserProfile):
    nodes    = list(G.nodes())
    n        = len(nodes)
    node_idx = {node: i for i, node in enumerate(nodes)}
 
    rows, cols, vals = [], [], []
    for u, v, data in G.edges(data=True):
        cost = cost_function(data, profile)
        if cost == float("inf"):
            cost = 1e6   # large but finite — preserves matrix structure
 
        i = node_idx[u]
        j = node_idx[v]
        w = 1.0 / cost   # conductance
 
        rows += [i, j, i, j]
        cols += [j, i, i, j]
        vals += [-w, -w, w, w]
 
    L = csr_matrix((vals, (rows, cols)), shape=(n, n))
    return L, nodes, node_idx
 
 
def _heat_method(G, source_node: int, profile: UserProfile,
                 t_factor: float = 1.0) -> dict:
    """
    Compute geodesic distances from source_node under the given profile.
    Returns {node_id: distance}.
    """
    L, nodes, node_idx = _build_laplacian(G, profile)
    n   = len(nodes)
    src = node_idx[source_node]
 
    costs  = [cost_function(d, profile)
              for _, _, d in G.edges(data=True)
              if cost_function(d, profile) < float("inf")]
    mean_c = float(np.mean(costs)) if costs else 1.0
 
    delta      = np.zeros(n)
    delta[src] = 1.0
    t          = t_factor * (mean_c ** 2)
 
    I = diags(np.ones(n), format="csr")
    u = spsolve(I + t * L, delta)
 
    div = np.zeros(n)
    for eu, ev, data in G.edges(data=True):
        cost = cost_function(data, profile)
        if cost == float("inf"):
            continue
        i    = node_idx[eu]
        j    = node_idx[ev]
        w    = 1.0 / cost
        grad = u[j] - u[i]
        norm = abs(grad)
        if norm < 1e-12:
            continue
        X      = -grad / norm
        div[i] += w * X
        div[j] -= w * X
 
    L_mod         = lil_matrix(L)
    div_mod       = div.copy()
    L_mod[src, :] = 0
    L_mod[src, src] = 1
    div_mod[src]  = 0
 
    phi  = spsolve(L_mod.tocsr(), div_mod)
    phi -= phi[src]
    phi  = np.abs(phi)
 
    return {nodes[i]: phi[i] for i in range(n)}
 
 
# ══════════════════════════════════════════════════════════════════════════════
# PATH RECOVERY  — gradient descent on φ
# ══════════════════════════════════════════════════════════════════════════════
 
def _recover_path(G, origin_node: int, dest_node: int,
                  distances: dict) -> list[int]:
    """
    Trace origin → destination by steepest descent of the geodesic field φ.
    Returns a list of node IDs (inclusive), or [] if unreachable.
    """
    path    = [dest_node]
    visited = {dest_node}
    current = dest_node
 
    for _ in range(len(G.nodes)):
        if current == origin_node:
            break
 
        phi_current  = distances.get(current, float("inf"))
        neighbours   = list(G.successors(current)) + list(G.predecessors(current))
 
        best_node = None
        best_phi  = phi_current
 
        for nb in neighbours:
            if nb in visited:
                continue
            edge_data = G.get_edge_data(current, nb) or G.get_edge_data(nb, current)
            if edge_data is None:
                continue
 
            phi_nb = distances.get(nb, float("inf"))
            if phi_nb < best_phi:
                best_phi  = phi_nb
                best_node = nb
 
        if best_node is None:
            print(f"  ⚠ Gradient descent stuck at node {current} "
                  f"(φ={phi_current:.2f}). Path may be incomplete.")
            break
 
        path.append(best_node)
        visited.add(best_node)
        current = best_node
 
    if current != origin_node:
        return []
 
    path.reverse()
    return path
 
 
# ══════════════════════════════════════════════════════════════════════════════
# RESULT DATA CLASS
# ══════════════════════════════════════════════════════════════════════════════
 
@dataclass
class QueryResult:
    origin_node      : int
    dest_node        : int
    travel_time_s    : float
    travel_time_min  : float
    profile_key      : str
    cache_hit        : bool
 
    def __str__(self):
        src = "cache" if self.cache_hit else "fresh solve"
        return (
            f"\n  Origin node  : {self.origin_node}"
            f"\n  Dest node    : {self.dest_node}"
            f"\n  Travel time  : {self.travel_time_min:.1f} min "
            f"({self.travel_time_s:.0f} s)"
            f"\n  Profile key  : {self.profile_key}"
            f"\n  Source       : {src}"
        )
 
 
# ══════════════════════════════════════════════════════════════════════════════
# ROUTING SERVICE
# ══════════════════════════════════════════════════════════════════════════════
 
class RoutingService:
    """
    On-demand safe-routing service backed by the Heat Method.
 
    Cache key: (origin_node, profile_key)
    Each unique origin + preference combination gets its own distance field.
    Subsequent queries from the same origin are O(1) dictionary lookups.
 
    Parameters
    ----------
    crime_csv     : Path to data.police.uk CSV
    cctv_csv      : Path to Bristol Open Data CCTV CSV
    lights_csv    : Path to Bristol Open Data street lighting CSV (optional)
    network_type  : "walk" (default) or "drive"
    t_factor      : Heat timestep scale (1.0 is a good default)
    graph_cache   : Path for the cached GraphML file
    """
 
    def __init__(
        self,
        crime_csv   : Optional[str] = None,
        cctv_csv    : Optional[str] = None,
        lights_csv  : Optional[str] = None,
        network_type: str   = "walk",
        t_factor    : float = 1.0,
        graph_cache : str   = "./bristol_graph.graphml",
    ):
        self.network_type = network_type
        self.t_factor     = t_factor
        self._G           = _build_graph(network_type, graph_cache)
        self._cache: dict[tuple, dict] = {}
 
        # Enrich edges with safety scores if CSV paths are provided
        if crime_csv and cctv_csv:
            self._G = enrich_graph_with_safety_data(
                self._G,
                crime_csv   = crime_csv,
                cctv_csv    = cctv_csv,
                lights_csv  = lights_csv,
            )
        else:
            print("  ⚠  No safety CSVs supplied. "
                  "Routing on OSM attributes only (lit= tag).\n"
                  "  Pass crime_csv= and cctv_csv= to RoutingService for full safe-routing.\n")
 
    # ── Core query ────────────────────────────────────────────────────────────
 
    def query(
        self,
        origin_lat: float, origin_lon: float,
        dest_lat  : float, dest_lon  : float,
        profile   : Optional[UserProfile] = None,
    ) -> QueryResult:
        if profile is None:
            profile = PROFILES["safest_day"]
 
        origin_node = ox.nearest_nodes(self._G, origin_lon, origin_lat)
        dest_node   = ox.nearest_nodes(self._G, dest_lon,   dest_lat)
 
        cache_key = (origin_node, profile.profile_key())
        cache_hit = cache_key in self._cache
 
        if not cache_hit:
            print(f"  Cache MISS — solving for profile '{profile.profile_key()}' ...")
            t0 = time.perf_counter()
            self._cache[cache_key] = _heat_method(
                self._G, origin_node, profile, self.t_factor
            )
            print(f"  Solve: {time.perf_counter() - t0:.2f} s — cached.\n")
 
        travel_time_s = self._cache[cache_key].get(dest_node, float("inf"))
 
        return QueryResult(
            origin_node    = origin_node,
            dest_node      = dest_node,
            travel_time_s  = travel_time_s,
            travel_time_min= travel_time_s / 60.0,
            profile_key    = profile.profile_key(),
            cache_hit      = cache_hit,
        )
 
    def batch_query(
        self,
        origin_lat  : float, origin_lon: float,
        destinations: list[tuple[float, float]],
        profile     : Optional[UserProfile] = None,
    ) -> list[QueryResult]:
        """One Heat Method solve, N destination lookups."""
        if profile is None:
            profile = PROFILES["safest_day"]
 
        dlat0, dlon0 = destinations[0]
        self.query(origin_lat, origin_lon, dlat0, dlon0, profile)
 
        return [
            self.query(origin_lat, origin_lon, dlat, dlon, profile)
            for dlat, dlon in destinations
        ]
 
    # ── Visualisation: static PNG ─────────────────────────────────────────────
 
    def visualise_from(
        self,
        origin_lat : float, origin_lon: float,
        dest_lat   : Optional[float] = None,
        dest_lon   : Optional[float] = None,
        profile    : Optional[UserProfile] = None,
        output_file: str = "bristol_safe_route.png",
    ):
        """
        Render a static heatmap (plasma palette) with optional route overlay.
        Node colours represent travel cost from the origin under the chosen profile.
        """
        if profile is None:
            profile = PROFILES["safest_day"]
 
        origin_node = ox.nearest_nodes(self._G, origin_lon, origin_lat)
        cache_key   = (origin_node, profile.profile_key())
 
        if cache_key not in self._cache:
            self.query(origin_lat, origin_lon, origin_lat, origin_lon, profile)
 
        distances = self._cache[cache_key]
 
        node_list = list(self._G.nodes(data=True))
        dist_vals = np.array([distances.get(n, np.nan) for n, _ in node_list])
        valid     = ~np.isnan(dist_vals)
        d_min, d_max = dist_vals[valid].min(), dist_vals[valid].max()
        norm      = mcolors.Normalize(vmin=d_min, vmax=d_max)
        cmap      = plt.cm.plasma
 
        node_colors = [cmap(norm(distances.get(n, d_max))) for n, _ in node_list]
 
        fig, ax = ox.plot_graph(
            self._G,
            node_color=node_colors, node_size=8,
            edge_color="#2a2a2a", edge_linewidth=0.4,
            bgcolor="#111111", show=False, close=False,
        )
 
        sx = self._G.nodes[origin_node]["x"]
        sy = self._G.nodes[origin_node]["y"]
        ax.scatter([sx], [sy], c="#ff4444", s=160, zorder=8,
                   marker="o", edgecolors="white", linewidths=1.2)
 
        legend_handles = [mpatches.Patch(color="#ff4444", label="Origin")]
 
        if dest_lat is not None and dest_lon is not None:
            dest_node   = ox.nearest_nodes(self._G, dest_lon, dest_lat)
            path_nodes  = _recover_path(self._G, origin_node, dest_node, distances)
 
            dx = self._G.nodes[dest_node]["x"]
            dy = self._G.nodes[dest_node]["y"]
            ax.scatter([dx], [dy], c="#00ff99", s=220, zorder=8,
                       marker="D", edgecolors="white", linewidths=1.2)
 
            if path_nodes:
                xs = [self._G.nodes[n]["x"] for n in path_nodes]
                ys = [self._G.nodes[n]["y"] for n in path_nodes]
                ax.plot(xs, ys, color="#006666", linewidth=5, zorder=5,
                        solid_capstyle="round")
                ax.plot(xs, ys, color="#00e5ff", linewidth=2.2, zorder=6,
                        solid_capstyle="round", label="Safe route")
 
                travel_s   = distances.get(dest_node, float("inf"))
                travel_min = travel_s / 60.0
                ax.text(
                    0.5, 0.04,
                    f"Safe route: {travel_min:.1f} min  ({travel_s:.0f} s)",
                    transform=ax.transAxes, ha="center", va="bottom",
                    fontsize=10, color="white",
                    bbox=dict(boxstyle="round,pad=0.4",
                              facecolor="#1a1a2e", edgecolor="#00e5ff",
                              linewidth=1.2, alpha=0.88),
                    zorder=10,
                )
                legend_handles.append(
                    mpatches.Patch(color="#00e5ff", label="Safe route"))
            else:
                ax.text(0.5, 0.04, "No path found between nodes",
                        transform=ax.transAxes, ha="center",
                        color="#ff6666", fontsize=9)
 
            legend_handles.append(
                mpatches.Patch(color="#00ff99", label="Destination"))
 
        ax.legend(handles=legend_handles, loc="upper left",
                  facecolor="#1a1a2e", edgecolor="#444",
                  labelcolor="white", fontsize=9)
 
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cb = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.01)
        cb.set_label("Routing cost from origin", color="white", fontsize=10)
        cb.ax.yaxis.set_tick_params(color="white")
        plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")
 
        ax.set_title(
            "Bristol — Safe Route (Crime · CCTV · Lighting)",
            color="white", fontsize=11, pad=10,
        )
 
        plt.tight_layout()
        plt.savefig(output_file, dpi=150, bbox_inches="tight",
                    facecolor="#111111")
        print(f"  Saved → {output_file}")
        plt.show()
 
    # ── Visualisation: interactive HTML ───────────────────────────────────────
 
    def visualise_interactive(
        self,
        origin_lat : float, origin_lon: float,
        dest_lat   : Optional[float] = None,
        dest_lon   : Optional[float] = None,
        profile    : Optional[UserProfile] = None,
        output_file: str = "bristol_safe_route.html",
    ):
        """
        Render an interactive Folium map:
          • Nodes coloured by routing cost (plasma palette)
          • Cyan route polyline with travel-time tooltip
          • Red origin marker, green destination marker
          • CCTV camera locations shown as blue circles (if enriched)
 
        Open the HTML in any browser — no server needed.
        """
        if profile is None:
            profile = PROFILES["safest_day"]
 
        origin_node = ox.nearest_nodes(self._G, origin_lon, origin_lat)
        cache_key   = (origin_node, profile.profile_key())
 
        if cache_key not in self._cache:
            self.query(origin_lat, origin_lon, origin_lat, origin_lon, profile)
 
        distances = self._cache[cache_key]
 
        dist_vals          = [v for v in distances.values() if v < float("inf")]
        d_min, d_max       = min(dist_vals), max(dist_vals)
 
        colormap = cm.LinearColormap(
            colors=["#0d0887", "#7e03a8", "#cc4778", "#f89540", "#f0f921"],
            vmin=d_min, vmax=d_max,
            caption="Routing cost from origin (lower = safer + faster)",
        )
 
        fmap = folium.Map(
            location=[origin_lat, origin_lon],
            zoom_start=14,
            tiles="CartoDB dark_matter",
        )
        colormap.add_to(fmap)
 
        # Node circles
        for node, data in self._G.nodes(data=True):
            dist = distances.get(node)
            if dist is None or dist == float("inf"):
                continue
            color = colormap(dist)
            folium.CircleMarker(
                location=(data["y"], data["x"]),
                radius=3,
                color=color, fill=True, fill_color=color,
                fill_opacity=0.85, weight=0,
                tooltip=f"Node {node} | cost {dist:.2f}",
            ).add_to(fmap)
 
        # Route polyline
        if dest_lat is not None and dest_lon is not None:
            dest_node  = ox.nearest_nodes(self._G, dest_lon, dest_lat)
            path_nodes = _recover_path(self._G, origin_node, dest_node, distances)
 
            if path_nodes:
                travel_s   = distances.get(dest_node, float("inf"))
                travel_min = travel_s / 60.0
                path_coords = [
                    (self._G.nodes[n]["y"], self._G.nodes[n]["x"])
                    for n in path_nodes
                ]
                # Glow effect
                folium.PolyLine(
                    path_coords, color="#006666", weight=8, opacity=0.5,
                ).add_to(fmap)
                folium.PolyLine(
                    path_coords, color="#00e5ff", weight=3, opacity=0.95,
                    tooltip=f"Safe route: {travel_min:.1f} min ({travel_s:.0f} s)",
                ).add_to(fmap)
 
            folium.Marker(
                location=(dest_lat, dest_lon),
                tooltip=f"Destination | {distances.get(dest_node, float('inf')):.2f} cost",
                icon=folium.Icon(color="green", icon="flag", prefix="fa"),
            ).add_to(fmap)
 
        # Origin marker
        folium.Marker(
            location=(origin_lat, origin_lon),
            tooltip="Origin",
            icon=folium.Icon(color="red", icon="circle", prefix="fa"),
        ).add_to(fmap)
 
        fmap.save(output_file)
        print(f"  Saved → {output_file}  (open in any browser)")
        return fmap
 
    # ── Utility ───────────────────────────────────────────────────────────────
 
    def cache_info(self) -> dict:
        return {
            "cached_origins": len(self._cache),
            "keys"          : list(self._cache.keys()),
        }
 
    def clear_cache(self):
        self._cache.clear()
        print("  Cache cleared.")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# BRISTOL LANDMARKS  (handy for quick testing)
# ══════════════════════════════════════════════════════════════════════════════
 
BRISTOL_LANDMARKS: dict[str, tuple[float, float]] = {
    "temple meads"   : (51.4493, -2.5815),
    "clifton village": (51.4579, -2.6233),
    "broadmead"      : (51.4578, -2.5908),
    "harbourside"    : (51.4497, -2.5985),
    "whiteladies rd" : (51.4660, -2.6075),
    "bedminster"     : (51.4384, -2.5919),
    "stokes croft"   : (51.4626, -2.5923),
    "cabot circus"   : (51.4589, -2.5832),
    "clifton down"   : (51.4692, -2.6158),
    "redland"        : (51.4713, -2.5983),
}
 
 
# ══════════════════════════════════════════════════════════════════════════════
# INTERACTIVE CLI
# ══════════════════════════════════════════════════════════════════════════════
 
def _interactive_cli():
    print("=" * 62)
    print("  Bristol Safe Routing Service — Interactive CLI")
    print("  Priority: Crime > CCTV > Lighting > Road speed")
    print("=" * 62)
 
    # ── Initialise service ────────────────────────────────────────────────────
    # Replace these paths with your actual CSV files:
    service = RoutingService(
        crime_csv  = "avon_somerset_crime.csv",   # data.police.uk
        cctv_csv   = "bristol_cctv.csv",          # opendata.bristol.gov.uk
        lights_csv = "bristol_street_lighting.csv",
        network_type = "walk",
    )
 
    print("\nKnown landmarks:")
    for name, (lat, lon) in BRISTOL_LANDMARKS.items():
        print(f"  {name:<20} ({lat:.4f}, {lon:.4f})")
 
    print("\nAvailable profiles:", list(PROFILES.keys()))
 
    def parse_location(s: str):
        s = s.strip().lower()
        if s in BRISTOL_LANDMARKS:
            return BRISTOL_LANDMARKS[s]
        try:
            lat, lon = map(float, s.split(","))
            return lat, lon
        except Exception:
            print(f"  ✗ Cannot parse '{s}'. Use a landmark name or 'lat,lon'.")
            return None
 
    while True:
        print("\n" + "─" * 40)
        origin_input = input("Origin (name or lat,lon) — or 'quit': ").strip()
        if origin_input.lower() in ("quit", "q", "exit"):
            break
 
        dest_input    = input("Destination (name or lat,lon)         : ").strip()
        profile_input = input(
            f"Profile [{'/'.join(PROFILES)}] (default: safest_day): "
        ).strip().lower() or "safest_day"
 
        origin  = parse_location(origin_input)
        dest    = parse_location(dest_input)
        profile = PROFILES.get(profile_input, PROFILES["safest_day"])
 
        if origin is None or dest is None:
            continue
 
        result = service.query(
            origin_lat=origin[0], origin_lon=origin[1],
            dest_lat  =dest[0],   dest_lon  =dest[1],
            profile   =profile,
        )
        print(result)
 
        viz = input("  Render map? (png / html / n) [n]: ").strip().lower()
        if viz == "png":
            service.visualise_from(
                origin[0], origin[1], dest[0], dest[1], profile=profile)
        elif viz == "html":
            service.visualise_interactive(
                origin[0], origin[1], dest[0], dest[1], profile=profile)
 
        print(f"  Cache: {service.cache_info()['cached_origins']} origin(s) stored.")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
 
if __name__ == "__main__":
    _interactive_cli()

ERROR! Session/line number was not unique in database. History logging moved to new session 24
  Bristol Safe Routing Service — Interactive CLI
  Priority: Crime > CCTV > Lighting > Road speed
  Saved to ./bristol_graph.graphml
  Graph ready: 38,422 nodes / 99,400 edges

  Enriching graph with safety data ...
    Loading crime data ...


FileNotFoundError: [Errno 2] No such file or directory: 'avon_somerset_crime.csv'